# PCB Anomaly Detection — PatchCore (pure PyTorch)

**의존성 충돌 0** — anomalib 안 쓰고 torch + torchvision + numpy 만 사용.

**입력**: 정상 PCB 사진 30~100장

**출력**:
- `patchcore_backbone.onnx` — ResNet18 피처 추출기 (Pi 에서 추론)
- `patchcore_coreset.npy` — 정상 패치 메모리뱅크 (Pi 에서 거리 계산)

**런타임**: Runtime → Change runtime type → **T4 GPU** 권장.

## 1. 라이브러리 확인

Colab 기본 설치된 것만 사용 — 추가 install 거의 없음.

In [ ]:
!pip install -q onnx onnxruntime
import torch, numpy, torchvision
print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('numpy:', numpy.__version__)
print('CUDA:', torch.cuda.is_available())

## 2. 정상 이미지 업로드

Cmd 누른 채로 30~100장 다중 선택.

In [ ]:
import os
from google.colab import files

NORMAL_DIR = '/content/pcb_normal'
os.makedirs(NORMAL_DIR, exist_ok=True)

uploaded = files.upload()
for name, content in uploaded.items():
    with open(os.path.join(NORMAL_DIR, name), 'wb') as f:
        f.write(content)

image_paths = sorted([
    os.path.join(NORMAL_DIR, f)
    for f in os.listdir(NORMAL_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])
print(f'정상 이미지 {len(image_paths)} 장 준비됨')

## 3. PatchCore 백본 정의 (ResNet18 의 layer2 + layer3 피처)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import numpy as np

IMG_SIZE = 256          # 입력 이미지 크기
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class PatchCoreBackbone(nn.Module):
    """ResNet18 의 layer2 + layer3 피처를 concat 해서 패치 임베딩 생성."""
    def __init__(self):
        super().__init__()
        bb = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.stem = nn.Sequential(bb.conv1, bb.bn1, bb.relu, bb.maxpool, bb.layer1)
        self.layer2 = bb.layer2
        self.layer3 = bb.layer3

    def forward(self, x):
        x = self.stem(x)
        f2 = self.layer2(x)                          # (B, 128, 32, 32)
        f3 = self.layer3(f2)                          # (B, 256, 16, 16)
        f3 = F.interpolate(f3, size=f2.shape[-2:],
                           mode='bilinear', align_corners=False)
        return torch.cat([f2, f3], dim=1)             # (B, 384, 32, 32)


backbone = PatchCoreBackbone().to(device).eval()
print('백본 준비 완료:', backbone.__class__.__name__)

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

## 4. 메모리 뱅크 구축 (정상 사진들로 패치 임베딩 수집)

각 이미지 → 32×32 = 1024 패치 × 384 차원 임베딩.

In [ ]:
from tqdm import tqdm

memory_bank = []
with torch.no_grad():
    for path in tqdm(image_paths, desc='피처 추출'):
        img = Image.open(path).convert('RGB')
        x = transform(img).unsqueeze(0).to(device)
        feat = backbone(x)                            # (1, 384, 32, 32)
        feat = feat.flatten(start_dim=2).permute(0, 2, 1).reshape(-1, feat.shape[1])
        memory_bank.append(feat.cpu())

memory_bank = torch.cat(memory_bank, dim=0)            # (N*1024, 384)
print(f'메모리 뱅크: {memory_bank.shape}, dtype={memory_bank.dtype}')

## 5. Coreset Subsampling (메모리 뱅크 축소)

정상 패치 임베딩을 그대로 두면 너무 크니까 무작위 30% 만 추출.
Pi 에서 검사 시 이 coreset 과 거리 계산.

In [ ]:
CORESET_RATIO = 0.3

n_total = len(memory_bank)
n_keep = max(100, int(n_total * CORESET_RATIO))
torch.manual_seed(42)
idx = torch.randperm(n_total)[:n_keep]
coreset = memory_bank[idx].contiguous()
print(f'coreset: {coreset.shape}  (메모리 뱅크 {n_total} → {n_keep} 개로 축소)')

## 6. 검증 — 정상 이미지 1장의 anomaly score 확인

정상이라 score 가 낮게 (~수십~수백 수준) 나와야 정상.

In [ ]:
def compute_anomaly(image_path):
    img = Image.open(image_path).convert('RGB')
    x = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = backbone(x)
    feat = feat.flatten(start_dim=2).permute(0, 2, 1).reshape(-1, feat.shape[1]).cpu()

    # 각 패치 → coreset 의 가장 가까운 점까지 거리
    dists = torch.cdist(feat, coreset)                 # (1024, n_keep)
    min_dists, _ = dists.min(dim=1)                    # (1024,)
    score = float(min_dists.max())                     # 이미지 전체 anomaly score
    heatmap = min_dists.reshape(32, 32).numpy()        # 패치별 점수 맵
    return score, heatmap


sample_score, sample_map = compute_anomaly(image_paths[0])
print(f'정상 샘플 score: {sample_score:.2f}')
print(f'heatmap shape: {sample_map.shape}, max: {sample_map.max():.2f}, mean: {sample_map.mean():.2f}')

## 7. 임계값 자동 추천

정상 이미지들의 score 분포로 평균 + 3σ 을 추천 임계값으로.

In [ ]:
scores = []
for path in tqdm(image_paths, desc='임계값 산출'):
    s, _ = compute_anomaly(path)
    scores.append(s)

scores = np.array(scores)
mean_s = scores.mean()
std_s = scores.std()
threshold = float(mean_s + 3 * std_s)

print(f'정상 score 평균: {mean_s:.2f}')
print(f'정상 score 표준편차: {std_s:.2f}')
print(f'추천 임계값 (평균 + 3σ): {threshold:.2f}')
print(f'→ 이 값보다 큰 score 면 결함 의심')

## 8. ONNX export — 백본만 (coreset 은 별도 .npy)

Pi 에서:
1. 이미지 → ONNX 백본 추론 → 패치 임베딩 (1024×384)
2. coreset.npy 와 cdist → score
3. score 가 임계값 초과 → 결함

In [ ]:
ONNX_PATH = '/content/patchcore_backbone.onnx'
CORESET_PATH = '/content/patchcore_coreset.npy'
META_PATH = '/content/patchcore_meta.json'

# 더미 입력으로 추적
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
torch.onnx.export(
    backbone,
    dummy,
    ONNX_PATH,
    input_names=['image'],
    output_names=['features'],
    dynamic_axes={'image': {0: 'batch'}, 'features': {0: 'batch'}},
    opset_version=14,
)
print(f'ONNX 저장: {ONNX_PATH} ({os.path.getsize(ONNX_PATH)/1024/1024:.1f} MB)')

np.save(CORESET_PATH, coreset.numpy().astype(np.float32))
print(f'Coreset 저장: {CORESET_PATH} ({os.path.getsize(CORESET_PATH)/1024/1024:.1f} MB)')

import json
meta = {
    'image_size': IMG_SIZE,
    'feature_dim': int(coreset.shape[1]),
    'coreset_size': int(coreset.shape[0]),
    'patch_grid': 32,
    'normalize_mean': [0.485, 0.456, 0.406],
    'normalize_std': [0.229, 0.224, 0.225],
    'threshold_mean_plus_3sigma': threshold,
    'normal_score_mean': float(mean_s),
    'normal_score_std': float(std_s),
    'n_normal_images': len(image_paths),
}
with open(META_PATH, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'Meta 저장: {META_PATH}')
print(json.dumps(meta, indent=2))

## 9. 다운로드 (3개 파일)

In [ ]:
for path in [ONNX_PATH, CORESET_PATH, META_PATH]:
    files.download(path)
print('다운로드 시작 — 3개 파일 모두')

## 10. Pi 로 배포

다운로드된 3개 파일을 Pi 의 `edge/weights/` 에 복사:

```bash
scp ~/Downloads/patchcore_backbone.onnx \
    ~/Downloads/patchcore_coreset.npy \
    ~/Downloads/patchcore_meta.json \
    pi@<pi_ip>:~/AiCapstoneV2/edge/weights/
```

Pi 쪽 추론 코드 (별도 작업):
1. `onnxruntime.InferenceSession(backbone.onnx)` 으로 피처 추출
2. `numpy.load('coreset.npy')` 로 메모리뱅크 로드
3. `np.linalg.norm(feat - coreset, axis=1).min()` 로 거리 계산
4. score > threshold → 결함

---

## 데이터 품질 체크리스트

- [ ] 모두 같은 PCB 종류
- [ ] 결함 0건 (사람이 검수)
- [ ] 1장당 다른 PCB
- [ ] 운영 카메라 / 거리 / 조명 으로 촬영
- [ ] 다양성 약간 (시간대·미세 위치·미세 회전)
- [ ] 30~100장 권장 (PatchCore 는 적은 데이터에 강함)